# Advanced Evaluation and Dataset Tools

This notebook covers the optional tools that are useful after the quickstart: generating custom SFT data, filtering it, flattening chat data into pretraining text, measuring tokenizer fertility, and comparing model outputs.

## Setup

Use the same Colab setup cell as the quickstart. If you are already in a cloned repo with dependencies installed, skip this cell.

In [ ]:
# 1. Clone the repo and enter the directory
!git clone https://github.com/mkhordoo/dusty-llm.git
%cd dusty-llm

# 2. Install uv (takes ~1 second)
!pip install uv

# 3. Install the project using uv into the Colab system environment (takes ~3 seconds)
!uv pip install --system -e .

# 4. Pull the TinyStories and Dusty datasets via your Make command
!make download-datasets

## Inspect the Dataset Files

The training loop expects raw data under `artifacts/datasets/` and tokenized datasets beside it.

In [ ]:
from pathlib import Path

for path in [
    Path("artifacts/datasets/tinystories_base.txt"),
    Path("artifacts/datasets/dusty_pretrain.txt"),
    Path("artifacts/datasets/dusty_sft.jsonl"),
]:
    if path.exists():
        print(f"OK {path}: {path.stat().st_size / 1024 / 1024:.2f} MB")
    else:
        print(f"MISSING {path}")

## Generate Your Own SFT Dataset

The SFT generator is OpenRouter-backed by default and reads the OpenRouter API key from `OPENAI_API_KEY`. The Makefile currently uses `DUSTY_MODEL=qwen/qwen3-235b-a22b-2507:floor` with `DUSTY_FALLBACK_MODEL=openai/gpt-oss-120b:floor`. In our runs, the Qwen model produced stronger Dusty-style rows, and the `:floor` suffix asks OpenRouter to route to the lowest-cost provider for that model.

You can try different generator models by overriding `DUSTY_MODEL` and `DUSTY_FALLBACK_MODEL`. If you change the assistant's personality, also update the generation prompt, categories, and seed examples in `dataset_generation/generate_sft_dataset_with_fallback.py`. The data generator should describe the exact character you want the trained model to imitate; otherwise the final SFT checkpoint will learn the old Dusty persona even if you rename the model.

Start small. The default full workflow asks for 500 examples per category, which is about 30k candidate rows across the 60 Dusty categories. With the current OpenRouter/Qwen default, generating that scale of data cost us less than $3, but prices and provider routing can change. This demo asks for 5 per category so you can check formatting, rejection rate, and style before spending money on a larger generation run.

In [ ]:
# Optional: set your OpenRouter key before running this cell.
# import os
# os.environ["OPENAI_API_KEY"] = "..."

!make dusty-generate-sft DUSTY_SFT_PER_CATEGORY=5 DUSTY_SFT_BATCH_SIZE=5

## Preview SFT Rows

Each JSONL row needs `category`, `user`, and `dusty`. The `category` field is metadata for filtering and balancing; training uses the user and assistant text.

In [ ]:
import json
from pathlib import Path

sft_path = Path("artifacts/datasets/dusty_sft.jsonl")
with sft_path.open("r", encoding="utf-8") as file:
    for index, line in zip(range(5), file):
        row = json.loads(line)
        print(json.dumps(row, indent=2)[:800])
        print()

## Filter and Balance SFT Data

Filtering removes answers that are too long for a tiny model and samples a category-balanced subset. Use a smaller target for experiments and a larger target for final training.

In [ ]:
!make dusty-filter-sft \
  DUSTY_SFT_FILTER_TARGET=200 \
  DUSTY_SFT_MAX_ANSWER_TOKENS=256 \
  DUSTY_SFT_FILTERED_OUT=artifacts/datasets/dusty_sft_200.jsonl


## Flatten SFT Data into ChatML Pretraining Text

Flattening is useful when you want the tokenizer or pretraining corpus to see the same ChatML boundary tokens used during SFT.

In [ ]:
!make dusty-flatten-sft-pretrain

from pathlib import Path

path = Path("artifacts/datasets/dusty_sft_chatml_pretrain.txt")
print(path)
print(path.read_text(encoding="utf-8")[:1000])

## Tokenizer Fertility

Fertility is tokens per whitespace-delimited word. For a tiny model, roughly `1.2` to `1.5` is usually a useful range. Higher values mean words are split heavily; lower values may spend too much of the model on embeddings.

Vocabulary size is part of the model's parameter budget. A larger vocabulary increases the embedding and output projection matrices, which can make the checkpoint larger without making the transformer layers more capable. Fertility testing asks a practical question: if we spend more parameters on vocabulary, do we actually reduce token count enough to justify it?

For Dusty, moving from about 4k vocabulary to 8k vocabulary did not produce a large enough fertility improvement to justify the extra parameters, so the default Dusty tokenizer stays around 4k tokens.

In [ ]:
!uv run python dataset_generation/tokenizer_fertility_test.py \
  --input artifacts/datasets/tinystories_base.txt \
  --training-files artifacts/datasets/tinystories_base.txt artifacts/datasets/dusty_sft_chatml_pretrain.txt \
  --lines 10000 \
  --vocab-sizes 2048 4096 8192


## Choose a Checkpoint by Generation

Loss is only a coarse signal for tiny character models. During training, step checkpoints such as `artifacts/checkpoints/dusty8m_sft_step_100.pt` let you evaluate intermediate model behavior. Generate the same fixed prompt set from several `CHECKPOINT_STEP` values and compare instruction following, persona consistency, repetition, and answer length.

Use the compute budget as a sanity check. The Chinchilla scaling rule of thumb is about 20 training tokens per model parameter. Dusty is about 8 million parameters, so a reasonable target is around 160 million seen tokens. With the default tiny dataset, that is roughly 1,900 iterations, or about 7 epochs. That is the region where we started choosing checkpoints qualitatively rather than simply stopping at the first low loss.

If you increase model size, increase the amount of training tokens. If you change the dataset size, the same token budget may happen at a different epoch count: a larger dataset reaches the target in fewer epochs, while a smaller dataset needs more repeated epochs and may overfit sooner.

When a step checkpoint behaves better than the final checkpoint, promote it by copying that file over the profile's default checkpoint path. After promotion, remove `CHECKPOINT_STEP` from the generation command; normal `make dusty-generate PROFILE=sft_dusty8m ...` will use the selected model.

In [ ]:
prompts = [
    "where are you?",
    "are you scared?",
    "what do you dream about?",
    "go charge",
]

checkpoint_steps = [50, 100, 150]

for step in checkpoint_steps:
    print("#" * 80)
    print("CHECKPOINT_STEP:", step)
    for prompt in prompts:
        print("=" * 80)
        print("PROMPT:", prompt)
        !make dusty-generate PROFILE=sft_dusty8m CHECKPOINT_STEP={step} PROMPT="{prompt}" TOP_P=0.9 TEMPERATURE=0.8

## Promote the Selected Checkpoint

Set `BEST_STEP` to the checkpoint step that produced the best qualitative behavior. This overwrites `artifacts/checkpoints/dusty8m_sft.pt`, which is the default checkpoint for the `sft_dusty8m` profile. Keep a backup if you still need the previous final checkpoint.

In [ ]:
BEST_STEP = 100

!cp artifacts/checkpoints/dusty8m_sft_step_{BEST_STEP}.pt artifacts/checkpoints/dusty8m_sft.pt

# This command intentionally omits CHECKPOINT_STEP. It now uses the promoted default checkpoint.
!make dusty-generate PROFILE=sft_dusty8m PROMPT="where are you?" TOP_P=0.9 TEMPERATURE=0.8